# 环节 03 · OpenAI 兼容探针与 TTFT 测法（配套 Notebook）

> 配套：[README.md](./README.md) §五 验证清单、[benchmark.md](./benchmark.md) §三 / §4.3
> 导航：[环节00](./环节00-总揽与环节导航.md)
> 定位：客户端**只换 `base_url`**；TTFT/TPOT 必须用 SSE 流式计时；思考型模型要认三个字段。**纯标准库。**
> §1–§4 不访问网络。§5 可选探测本机端口，没起服务就跳过。

| 本 Notebook | 手册 | 验证什么 |
|---|---|---|
| §1 四家端口 | README §二 | 客户端契约只有 base_url + 模型名 |
| §2 跑通清单 | README §五 | `/v1/models` 与 chat.completions |
| §3 SSE 三字段 | benchmark 坑 6 | `content` / `reasoning` / `reasoning_content` |
| §4 TTFT/TPOT + nonce | benchmark §三 / 陷阱 4 | 前缀缓存会把 TTFT 打到 1/30 |
| §5 可选本机探测 | — | 11434 / 8080 / 8081 / 1234 |


## 1. 四家都是同一份协议


In [ ]:
RUNTIMES = {
    "llama.cpp":  {"base": "http://127.0.0.1:8081/v1", "note": "llama serve --port 8081"},
    "Ollama":     {"base": "http://127.0.0.1:11434/v1", "note": "ollama serve"},
    "LM Studio":  {"base": "http://127.0.0.1:1234/v1",  "note": "GUI 里 Start Server"},
    "MLX":        {"base": "http://127.0.0.1:8080/v1", "note": "mlx_lm.server --port 8080"},
}

print("客户端只改这两行：\n")
for name, cfg in RUNTIMES.items():
    print(f"  # {cfg['note']}")
    print(f"  base_url = {cfg['base']!r}")
    print()
print("请求体字段（model / messages / stream / temperature）四家通用。")
print("Ollama 另有原生 /api/generate；要横向对比请走 /v1，别混。")


## 2. 「跑通了」的最小请求体

验证清单五条里，前三条都可以用同一份 JSON。下面只**构造**请求，不发送。


In [ ]:
import json

def chat_body(model, prompt, stream, nonce=None):
    text = prompt if not nonce else f"[nonce:{nonce}]\n{prompt}"
    return {
        "model": model,
        "messages": [{"role": "user", "content": text}],
        "stream": stream,
        "temperature": 0,
        "max_tokens": 64,
    }

print("非流式：POST {base}/chat/completions")
print(json.dumps(chat_body("qwen3:0.6b", "用一句话介绍你自己", stream=False), ensure_ascii=False, indent=2))
print("\n流式：同一份，stream=true。前端打字机 = 消费 SSE 的 delta。")


## 3. SSE：必须同时认三个字段

只认 `delta.content` 会把思考型模型的输出判成空——MLX 走 `reasoning`，llama.cpp 走 `reasoning_content`。


In [ ]:
def delta_text(chunk: dict) -> str:
    delta = (chunk.get("choices") or [{}])[0].get("delta") or {}
    for key in ("content", "reasoning", "reasoning_content"):
        val = delta.get(key)
        if val:
            return val
    return ""


def parse_sse_line(line):
    line = line.strip()
    if not line.startswith("data:"):
        return None
    payload = line[5:].strip()
    if payload == "[DONE]":
        return None
    return json.loads(payload)


# 模拟三条流：普通 / MLX 思考 / llama.cpp 思考
STREAMS = {
    "普通 content": [
        'data: {"choices":[{"delta":{"content":"你好"}}]}',
        'data: {"choices":[{"delta":{"content":"。"}}]}',
        "data: [DONE]",
    ],
    "MLX reasoning": [
        'data: {"choices":[{"delta":{"reasoning":"先想一下"}}]}',
        'data: {"choices":[{"delta":{"content":"答案"}}]}',
        "data: [DONE]",
    ],
    "llama.cpp reasoning_content": [
        'data: {"choices":[{"delta":{"reasoning_content":"hmm"}}]}',
        'data: {"choices":[{"delta":{"content":"42"}}]}',
        "data: [DONE]",
    ],
}

def consume(lines):
    parts = []
    for line in lines:
        chunk = parse_sse_line(line)
        if chunk is None:
            continue
        t = delta_text(chunk)
        if t:
            parts.append(t)
    return "".join(parts)

for name, lines in STREAMS.items():
    only_content = []
    for line in lines:
        chunk = parse_sse_line(line)
        if not chunk:
            continue
        c = ((chunk.get("choices") or [{}])[0].get("delta") or {}).get("content") or ""
        if c:
            only_content.append(c)
    print(f"{name:<32} 只认 content={''.join(only_content)!r:8}  三字段={consume(lines)!r}")


## 4. TTFT / TPOT：流式计时 + nonce 打前缀缓存

```
t0 = perf_counter()
for 每个有文本的 SSE 块:
    若 ttft 还是 None: ttft = now - t0
tpot = (total - ttft) / (n_tokens - 1)
```

本机坑：Ollama 同一 prompt 第二次 TTFT **868 ms → 23 ms**（命中前缀缓存）。对比时把 **变化的 nonce 放在 prompt 开头**（放结尾没用）。


In [ ]:
import time
from itertools import count

_fake_clock = [0.0]


def fake_sleep(ms: float) -> None:
    _fake_clock[0] += ms / 1000.0


def now() -> float:
    return _fake_clock[0]


def measure_stream(events: list[tuple[float, str]]) -> dict:
    """events = [(delay_ms_before_this_chunk, text), ...]"""
    _fake_clock[0] = 0.0
    t0 = now()
    ttft = None
    n = 0
    for delay, text in events:
        fake_sleep(delay)
        if text:
            n += 1
            if ttft is None:
                ttft = now() - t0
    total = now() - t0
    tpot = (total - ttft) / (n - 1) if n > 1 else None
    return {"ttft_ms": ttft * 1000, "tpot_ms": None if tpot is None else tpot * 1000, "n": n, "total_ms": total * 1000}


cold = [(50, "你"), (4, "好"), (4, "。")]          # 首包慢 = 冷启动 / 未命中缓存
hot  = [(2, "你"), (4, "好"), (4, "。")]           # 前缀命中
print("冷态 ", {k: (round(v, 1) if isinstance(v, float) else v) for k, v in measure_stream(cold).items()})
print("热态 ", {k: (round(v, 1) if isinstance(v, float) else v) for k, v in measure_stream(hot).items()})

nonce = next(count(1))
body = chat_body("qwen3:0.6b", "用一句话介绍你自己", stream=True, nonce=f"{int(time.time())}-{nonce}")
print("\n防缓存：nonce 必须在 messages[0].content 的开头")
print(body["messages"][0]["content"][:40], "...")
print("\n本机对照（短 prompt ≈166 tok → ≈62 tok）：")
print("  llama-server TTFT 18–33 ms, TPOT ≈ 3.8 ms")
print("  Ollama       TTFT 50–70 ms, TPOT ≈ 3.8 ms")
print("  MLX server   TTFT 116–150 ms, TPOT ≈ 3.5 ms  ← 裸框架最快，套 HTTP 后 TTFT 反而最高")


## 5. 可选：探测本机是否已经起了服务

没起任何东西时这格只会打印「未发现」。**不要**在 notebook 里自动 `ollama run`——那是手册的事。


In [ ]:
import urllib.error
import urllib.request

PORTS = [
    (11434, "Ollama"),
    (8080, "MLX mlx_lm.server"),
    (8081, "llama serve"),
    (1234, "LM Studio"),
]


def probe(port: int, timeout: float = 0.4) -> str:
    url = f"http://127.0.0.1:{port}/v1/models"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            raw = resp.read().decode("utf-8", errors="replace")
        data = json.loads(raw)
        ids = [m.get("id") or m.get("name") for m in data.get("data") or []]
        return "OK  models=" + ",".join(ids[:6])
    except urllib.error.HTTPError as e:
        return f"HTTP {e.code}（服务在，但 /v1/models 未开）"
    except Exception as e:
        return f"未发现 ({type(e).__name__})"


found = False
for port, name in PORTS:
    msg = probe(port)
    if msg.startswith("OK") or msg.startswith("HTTP"):
        found = True
    print(f"  :{port:<5} {name:<22} {msg}")
if not found:
    print("\n本机没有可用的 OpenAI 兼容端口。按手册起一个再重跑本格：")
    print("  ollama serve          # :11434")
    print("  llama serve -m <gguf> --port 8081 -ngl 99 -c 4096")
    print("  mlx_lm.server --model <mlx> --port 8080")
